# CPA Screening — End-to-End Runner

This notebook runs the whole Phase 1 Day 1 pipeline on Colab:
1. Mount Drive (optional persistent cache)
2. Clone the repo and install deps
3. Build the consolidated dataset + print audit
4. Train the Random Forest baseline
5. Display results and zip artifacts for download

Phase 1 Day 1 only includes the RF baseline against DOLMEN (and Higgins, if you've supplied the hand-curated CSVs). ChemBERTa+LoRA arrives in Day 2 after PAUSE POINT 1.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d cpa-screening ] || git clone https://github.com/cmendoza1031/cpa-screening.git
%cd /content/cpa-screening
!git pull --ff-only

In [ ]:
!pip install --quiet -r requirements.txt

In [ ]:
# Optional: persist data/results across Colab sessions via Drive.
import os, pathlib
DRIVE_CACHE = pathlib.Path('/content/drive/MyDrive/cpa-screening')
if DRIVE_CACHE.parent.exists():
    DRIVE_CACHE.mkdir(exist_ok=True)
    for sub in ('data', 'results'):
        src = pathlib.Path(sub)
        dst = DRIVE_CACHE / sub
        dst.mkdir(parents=True, exist_ok=True)
        if src.is_symlink():
            src.unlink()
        elif src.exists() and not any(src.iterdir()):
            src.rmdir()
        if not src.exists():
            os.symlink(dst.resolve(), src)
    print('Linked data/ and results/ to', DRIVE_CACHE)
else:
    print('Drive not mounted; using ephemeral storage')

## 2. Build the dataset

First run will download DOLMEN raw CSVs and Tox21 (~5 MB), then map the FDA IID CAS list to SMILES via PubChem (slow first run, ~1.5k entries; cached after).

If you have the hand-curated Higgins CSVs (Jan 2025 + Dec 2025), drop them at `data/raw/higgins_jan2025.csv` and `data/raw/higgins_dec2025.csv` BEFORE running this cell. If absent, templates will be auto-created and the rest of the pipeline keeps going (DOLMEN-only IRI training).

In [ ]:
# Skip FDA on the first dev run if you want fast iteration.
# !python -m src.data --skip-fda --skip-tox21
!python -m src.data

In [ ]:
import json, pathlib
audit = json.loads(pathlib.Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2)[:2000])

## 3. Train Random Forest baseline

In [ ]:
!python -m src.train --model rf --seed 0

In [ ]:
import pandas as pd
df = pd.read_csv('results/results_table.csv')
df

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('results/figures').glob('*.png')):
    print(p)
    display(Image(str(p)))

## 4. Bundle results for download

In [ ]:
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip

In [ ]:
from google.colab import files
files.download('results.zip')

---
## PAUSE POINT 1

Report back to the developer with:
- Compound counts per task from the audit cell above (look at `unique_compounds` and `per_task`)
- RF test metrics from `results_table.csv` (rows where `split == "test"`)
- Any errors or PubChem failure counts (`audit.pubchem_cache.n_misses`, plus `data/.cache/pubchem_failures.txt` contents)

Phase 1 Day 2 (ChemBERTa+LoRA) starts after confirmation.